# State-classification workflow

This notebook covers both binary and ternary readout classification.
Start with a standard ge experiment for the 2-state classifier, then move
to `ge-ef-cr` mode when you want to resolve `|g>`, `|e>`, and `|f>`.


## 1. Build a 2-state classifier

Start from the ordinary ge configuration and collect the readout clusters
for `|g>` and `|e>`.


In [ ]:
import numpy as np

import qubex as qx

exp_ge = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

Q0 = exp_ge.qubit_labels[0]
print("target qubit:", Q0)

## 2. Connect and prepare the ge pulse set


In [ ]:
exp_ge.connect()

waveform_result = exp_ge.check_waveform(targets=Q0, n_shots=512, plot=True)
rabi_result = exp_ge.obtain_rabi_params(targets=Q0, n_shots=512, plot=True)
hpi_result = exp_ge.calibrate_hpi_pulse(targets=Q0, n_shots=512, plot=True)

## 3. Measure the 2-state clusters and build the classifier


In [ ]:
binary_distributions = exp_ge.measure_state_distribution(
    targets=Q0,
    n_states=2,
    n_shots=2048,
    plot=True,
)

binary_classifier = exp_ge.build_classifier(
    targets=Q0,
    n_states=2,
    save_classifier=False,
    n_shots=4096,
    plot=True,
)

## 4. Re-open the experiment in `ge-ef-cr` mode for 3-state classification


In [ ]:
exp_gef = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    qubits=[0],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

exp_gef.connect()
Q0_gef = exp_gef.qubit_labels[0]

## 5. Prepare the EF pulse set

Ternary classification assumes you can also prepare the `|f>` manifold in
a controlled way.


In [ ]:
ge_rabi_result = exp_gef.obtain_rabi_params(targets=Q0_gef, n_shots=512, plot=True)
ge_hpi_result = exp_gef.calibrate_hpi_pulse(targets=Q0_gef, n_shots=512, plot=True)
ef_rabi_result = exp_gef.obtain_ef_rabi_params(
    targets=Q0_gef,
    time_range=np.arange(0, 201, 4),
    n_shots=512,
    plot=True,
)
ef_hpi_result = exp_gef.calibrate_ef_hpi_pulse(
    targets=Q0_gef,
    n_rotations=1,
    n_shots=512,
    plot=True,
)

## 6. Measure the 3-state clusters and build the classifier


In [ ]:
ternary_distributions = exp_gef.measure_state_distribution(
    targets=Q0_gef,
    n_states=3,
    n_shots=2048,
    plot=True,
)

ternary_classifier = exp_gef.build_classifier(
    targets=Q0_gef,
    n_states=3,
    save_classifier=False,
    n_shots=4096,
    plot=True,
)

## 7. Save the classifier artifacts only when you are ready


In [ ]:
# exp_ge.build_classifier(targets=Q0, n_states=2, save_classifier=True)
# exp_gef.build_classifier(targets=Q0_gef, n_states=3, save_classifier=True)
